##### sample dataset

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Delta_Project").getOrCreate() 
data = [ 
        (1, "C001", "Laptop", 50000), 
        (2, "C002", "Mobile", 15000), 
        (3, "C003", "Tablet", 20000), 
        (4, "C004", "Laptop", 55000) 
    ] 
columns = ["id", "customer_id", "product", "amount"] 
df = spark.createDataFrame(data, columns)

##### store as delta table

In [0]:
df.write.format("delta")\
  .mode("overwrite")\
    .save("/Volumes/workspace/default/day6_deltatable")

##### read delta table

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/day6_deltatable")
df.display()


##### insert new data

In [0]:
new_data = [(5, "C005", "Laptop", 60000), (6, "C006", "Tablet", 25000)]
new_df = spark.createDataFrame(new_data,columns)

new_df.write.format("delta")\
  .mode("append").save("/Volumes/workspace/default/day6_deltatable")

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/day6_deltatable")
df.display()


##### update


In [0]:
from delta.tables import DeltaTable
delta_table=DeltaTable.forPath(spark,"/Volumes/workspace/default/day6_deltatable")

delta_table.update(
  condition="id=5",
  set = {"amount":"80000"}
)

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/day6_deltatable")
df.display()


##### delete

In [0]:
delta_table.delete("id=6")

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/day6_deltatable")
df.display()


##### Merge into

In [0]:
updates = [
    (3, "C003", "Tablet", 22000),  # update
    (6, "C006", "Watch", 8000)     # insert
]

columns = ["id", "customer_id", "product", "amount"]

updates_df = spark.createDataFrame(updates, columns)
updates_df.display()

In [0]:
#  If id exists → UPDATE
#  If id not exists → INSERT
#  This is UPSERT
delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "customer_id": "source.customer_id",
    "product": "source.product",
    "amount": "source.amount"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "customer_id": "source.customer_id",
    "product": "source.product",
    "amount": "source.amount"
}).execute()


In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/day6_deltatable")
df.display()

#### Schema Evolution

In [0]:
new_data2 = [
    (7, "C007", "Headphones", 5000, "India"),
    (8, "C008", "Camera", 35000, "USA")
]

new_columns = ["id", "customer_id", "product", "amount", "country"]

new_df = spark.createDataFrame(new_data2, new_columns)

In [0]:
# On serverless compute, use .option("mergeSchema", "true") in write operations
# instead of spark.databricks.delta.schema.autoMerge.enabled

new_df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/day6_deltatable")

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/day6_deltatable")
df.printSchema()
df.display()

In [0]:
delta_table.history().display()

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/day6_deltatable")
df_old.display()

##### Restore table

In [0]:
delta_table.restoreToVersion(0)